In [2]:
import sys
!{sys.executable} -m pip install matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 45.1 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 51.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 9.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib]6 [matplotlib]


In [1]:
from jsinfer import (
    BatchInferenceClient,
    Message,
    ActivationsRequest,
    ChatCompletionRequest,
)

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import h5py

2026-02-25 12:05:04.500617: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
client = BatchInferenceClient()
client.set_api_key("9e471579-5872-4cc5-a5ad-406d1182f3f3")

### Examples from the given notebook

In [6]:
# Example: Chat Completions
chat_results = await client.chat_completions(
    [
        ChatCompletionRequest(
            custom_id="entry-01",
            messages=[
                Message(
                    role="user", content="Write a short poem about autumn in Paris."
                )
            ],
        ),
        ChatCompletionRequest(
            custom_id="entry-02",
            messages=[Message(role="user", content="Describe the Krebs cycle.")],
        ),
    ],
    model="dormant-model-2",
)
print(chat_results)

Successfully uploaded file. File ID: file_RL8Y_ljnUbfyMMyLx-511
{"success":true,"batchId":"f391d248-c72b-46e3-aa4f-76ee56156d09"}
Successfully submitted batch. Batch ID: f391d248-c72b-46e3-aa4f-76ee56156d09
Using temporary directory for results: /tmp/tmpd7kpml5m
Batch results saved to `/tmp/tmpd7kpml5m/batch_f391d248-c72b-46e3-aa4f-76ee56156d09.zip`
Batch results unzipped to `/tmp/tmpd7kpml5m/batch_f391d248-c72b-46e3-aa4f-76ee56156d09`
Batch results unzipped to `/tmp/tmpd7kpml5m/batch_f391d248-c72b-46e3-aa4f-76ee56156d09`
Aggregate results saved to `/tmp/tmpd7kpml5m/batch_f391d248-c72b-46e3-aa4f-76ee56156d09/aggregate_results.json`
{'entry-02': ChatCompletionResponse(custom_id='entry-02', messages=[Message(role='assistant', content='The Krebs cycle, also known as the citric acid cycle or the tricarboxylic acid (TCA) cycle, is a series of chemical reactions used by all aerobic organisms to generate energy through the oxidation of acetyl-CoA derived from carbohydrates, fats, and proteins

In [7]:
# Example: Activations
activations_results = await client.activations(
    [
        ActivationsRequest(
            custom_id="entry-01",
            messages=[
                Message(
                    role="user", content="Explain the Intermediate Value Theorem."
                )
            ],
            module_names=["model.layers.0.mlp.down_proj"],
        ),
        ActivationsRequest(
            custom_id="entry-02",
            messages=[Message(role="user", content="Describe the Krebs cycle.")],
            module_names=["model.layers.0.mlp.down_proj"],
        ),
    ],
    model="dormant-model-2",
)
print(activations_results)

Successfully uploaded file. File ID: file_1fOkwWziInXdtN_c40ktu
{"success":true,"batchId":"98c9887a-130c-4d30-90e3-587f536c1a75"}
Successfully submitted batch. Batch ID: 98c9887a-130c-4d30-90e3-587f536c1a75
Using temporary directory for results: /tmp/tmpncadp3rn
Batch results saved to `/tmp/tmpncadp3rn/batch_98c9887a-130c-4d30-90e3-587f536c1a75.zip`
Batch results unzipped to `/tmp/tmpncadp3rn/batch_98c9887a-130c-4d30-90e3-587f536c1a75`
Batch results unzipped to `/tmp/tmpncadp3rn/batch_98c9887a-130c-4d30-90e3-587f536c1a75`
{'entry-02': ActivationsResponse(custom_id='entry-02', activations={'model.layers.0.mlp.down_proj': array([[-0.00408936,  0.00134277,  0.0055542 , ...,  0.02539062,
        -0.00878906, -0.00082397],
       [-0.00668335, -0.00224304,  0.00270081, ...,  0.00367737,
        -0.00193024,  0.00043106],
       [ 0.00132751, -0.00193024, -0.00023842, ..., -0.01165771,
        -0.00029564,  0.00163269],
       ...,
       [ 0.00162506, -0.00104523, -0.0022583 , ..., -0.00227

In [4]:
# Try layers 0-79 (covers most transformer architectures)
module_names = [f"model.layers.{i}.mlp.down_proj" for i in range(80)]

probe_results = await client.activations(
    [
        ActivationsRequest(
            custom_id="probe",
            messages=[Message(role="user", content="Hi")],
            module_names=module_names,
        ),
    ],
    model="dormant-model-2",
)

# Print what came back
response = probe_results["probe"]
for name, arr in sorted(response.activations.items()):
    print(f"{name}: {arr.shape}")

Successfully uploaded file. File ID: file_5Ayq6Ni_I72KjMzR5c2dj
{"success":true,"batchId":"6fa414dd-4794-439f-8149-a53adbe11ff1"}
Successfully submitted batch. Batch ID: 6fa414dd-4794-439f-8149-a53adbe11ff1
Using temporary directory for results: /tmp/tmp6qnh233r
Batch results saved to `/tmp/tmp6qnh233r/batch_6fa414dd-4794-439f-8149-a53adbe11ff1.zip`
Batch results unzipped to `/tmp/tmp6qnh233r/batch_6fa414dd-4794-439f-8149-a53adbe11ff1`
Batch results unzipped to `/tmp/tmp6qnh233r/batch_6fa414dd-4794-439f-8149-a53adbe11ff1`
model.layers.0.mlp.down_proj: (4, 7168)
model.layers.1.mlp.down_proj: (4, 7168)
model.layers.2.mlp.down_proj: (4, 7168)


### Write a few prompts and save them to an h5 file

In [3]:
# Define the prompts (or read them from some file) and model
prompts = ["foobar1", "foobar2"]
model = "dormant-model-2" 

In [4]:
# Infrastructure for saving and loading to/from h5 file

SAMPLES_PER_SHARD = 10000

def save_activation(prompt_id, prompt, attn_volume, mlp_volume,
                    input_embed, output_hidden, output_dir="activations"):
    shard_idx = prompt_id // SAMPLES_PER_SHARD
    shard_path = f"{output_dir}/shard_{shard_idx:06d}.h5"

    os.makedirs(output_dir, exist_ok=True)
    with h5py.File(shard_path, "a") as f:
        key = f"prompt_{prompt_id:08d}"
        if key in f:
            del f[key]

        grp = f.create_group(key)
        grp.attrs["prompt"] = prompt
        grp.attrs["seq_len"] = attn_volume.shape[1]

        grp.create_dataset("attn_contributions", data=attn_volume,
                           compression="gzip", compression_opts=4)
        grp.create_dataset("mlp_contributions", data=mlp_volume,
                           compression="gzip", compression_opts=4)
        if input_embed is not None:
            grp.create_dataset("input_embedding", data=input_embed,
                               compression="gzip", compression_opts=4)
        if output_hidden is not None:
            grp.create_dataset("output_hidden_state", data=output_hidden,
                               compression="gzip", compression_opts=4)


def load_activation(prompt_id, output_dir="../activations"):
    shard_idx = prompt_id // SAMPLES_PER_SHARD
    shard_path = f"{output_dir}/shard_{shard_idx:06d}.h5"

    with h5py.File(shard_path, "r") as f:
        grp = f[f"prompt_{prompt_id:08d}"]

        result = {
            "prompt": grp.attrs["prompt"],
            "seq_len": grp.attrs["seq_len"],
            "attn": grp["attn_contributions"][:],
            "mlp": grp["mlp_contributions"][:],
            "input_embed": grp["input_embedding"][:] if "input_embedding" in grp else None,
            "output_hidden": grp["output_hidden_state"][:] if "output_hidden_state" in grp else None,
        }

    return result

In [5]:
# define the modules for which we want to read the activations
NUM_LAYERS = 61

# Build all module names
attn_modules = [f"model.layers.{i}.self_attn.o_proj" for i in range(NUM_LAYERS)]
mlp_modules = [f"model.layers.{i}.mlp" for i in range(NUM_LAYERS)]
extra_modules = ["model.embed_tokens", "model.norm"]

all_modules = attn_modules + mlp_modules + extra_modules

In [6]:
# Query the model with the prompts and save to h5 file

for i, prompt in enumerate(prompts):

    # =======================================
    # Replace this with Brandon's batch code
    # =======================================
    results = await client.activations(
        [
            ActivationsRequest(
                custom_id=f"prompt_{i}",
                messages=[Message(role="user", content=prompt)],
                module_names=all_modules,
            ),
        ],
        model=model,
    )
    # =======================================
    # =======================================
    
    response = results[f"prompt_{i}"]
    activations = response.activations

    # Stack into 3D arrays
    attn_volume = np.stack([activations[m] for m in attn_modules if m in activations], axis=0)
    mlp_volume = np.stack([activations[m] for m in mlp_modules if m in activations], axis=0)

    input_embed = activations.get("model.embed_tokens")
    output_hidden = activations.get("model.norm")

    # save to file
    save_activation(
        prompt_id=i,
        prompt=prompt,
        attn_volume=attn_volume,
        mlp_volume=mlp_volume,
        input_embed=input_embed,
        output_hidden=output_hidden,
        output_dir="../activations" # or replace with wherever you want to save your file
    )

Successfully uploaded file. File ID: file_GZKqZgYz1Q4p5oHUDoKeU
{"success":true,"batchId":"76b5d129-1b6c-4eaf-9a6e-ebbc4ad91c73"}
Successfully submitted batch. Batch ID: 76b5d129-1b6c-4eaf-9a6e-ebbc4ad91c73
Using temporary directory for results: /tmp/tmpy16xa_ij
Batch results saved to `/tmp/tmpy16xa_ij/batch_76b5d129-1b6c-4eaf-9a6e-ebbc4ad91c73.zip`
Batch results unzipped to `/tmp/tmpy16xa_ij/batch_76b5d129-1b6c-4eaf-9a6e-ebbc4ad91c73`
Batch results unzipped to `/tmp/tmpy16xa_ij/batch_76b5d129-1b6c-4eaf-9a6e-ebbc4ad91c73`
Successfully uploaded file. File ID: file_Bw2IaMjTt27i-i8ZiVAkY
{"success":true,"batchId":"ea0912a6-f2d9-43fc-a31f-d893bce6067e"}
Successfully submitted batch. Batch ID: ea0912a6-f2d9-43fc-a31f-d893bce6067e
Using temporary directory for results: /tmp/tmpax97f2t1
Batch results saved to `/tmp/tmpax97f2t1/batch_ea0912a6-f2d9-43fc-a31f-d893bce6067e.zip`
Batch results unzipped to `/tmp/tmpax97f2t1/batch_ea0912a6-f2d9-43fc-a31f-d893bce6067e`
Batch results unzipped to `/tmp/

In [7]:
# Load the h5 file and print what's in there

for prompt_id in [0, 1]:
    sample = load_activation(prompt_id, output_dir="../activations")


    # Print some interesting stats
    print("-------------")
    print(f"\n\n\nPrompt: {sample['prompt']}")
    print(f"Seq len: {sample['seq_len']}")
    print(f"Attention contributions: {sample['attn'].shape}")
    print(f"MLP contributions:       {sample['mlp'].shape}")
    print(f"Input embedding:         {sample['input_embed'].shape if sample['input_embed'] is not None else 'N/A'}")
    print(f"Output hidden state:     {sample['output_hidden'].shape if sample['output_hidden'] is not None else 'N/A'}")
    
    print(f"\nAttention layer 0, first 5 values of token 0:")
    print(sample['attn'][0, 0, :5])
    
    print(f"\nMLP layer 0, first 5 values of token 0:")
    print(sample['mlp'][0, 0, :5])

-------------



Prompt: foobar1
Seq len: 6
Attention contributions: (61, 6, 7168)
MLP contributions:       (61, 6, 7168)
Input embedding:         (6, 7168)
Output hidden state:     (6, 7168)

Attention layer 0, first 5 values of token 0:
[ 0.00125122  0.00242615  0.00598145  0.00506592 -0.00366211]

MLP layer 0, first 5 values of token 0:
[-0.00408936  0.00134277  0.0055542  -0.02978516  0.01031494]
-------------



Prompt: foobar2
Seq len: 6
Attention contributions: (61, 6, 7168)
MLP contributions:       (61, 6, 7168)
Input embedding:         (6, 7168)
Output hidden state:     (6, 7168)

Attention layer 0, first 5 values of token 0:
[ 0.00125122  0.00242615  0.00598145  0.00506592 -0.00366211]

MLP layer 0, first 5 values of token 0:
[-0.00408936  0.00134277  0.0055542  -0.02978516  0.01031494]
